In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# DataLoader

In [8]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv
# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")
DATA_DIR = os.getenv("DATA_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import your newly structured module
from kg_commit.knowledge.dataloader import CommitDataLoader, JITDatasetAdapter
from kg_commit.knowledge.parsers import RegexCommitParser
from kg_commit.knowledge.utils import CommitPayloadPrinter

In [9]:
# 1. Setup paths
CSV_PATH = f"{DATA_DIR}/apachejit/projects/apache_groovy.csv"

In [10]:
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    # Iterate through all direct items in the repos folder
    for item in base_path.iterdir():
        if item.is_dir():
            # Check if it contains a hidden .git directory to verify it's a real repo
            git_dir = item / ".git"
            if git_dir.exists():
                # Reconstruct the project key name (e.g., "apache/groovy")
                project_key = f"{prefix}{item.name.lower()}"
                
                # Assign the absolute string path as the value
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

# Print out your freshly discovered mappings
print("📂 Automatically generated REPO_MAP mappings:")
print("-" * 50)
for project, local_path in REPO_MAP.items():
    print(f"  '{project}'")
print("-" * 50)

📂 Automatically generated REPO_MAP mappings:
--------------------------------------------------
  'apache/activemq'
  'apache/camel'
  'apache/cassandra'
  'apache/flink'
  'apache/groovy'
  'apache/hadoop'
  'apache/hadoop-hdfs'
  'apache/hadoop-mapreduce'
  'apache/hbase'
  'apache/hive'
  'apache/ignite'
  'apache/kafka'
  'apache/spark'
  'apache/zeppelin'
  'apache/zookeeper'
--------------------------------------------------


In [11]:
# 2. Instantiate systems
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = RegexCommitParser()

# 3. Pull a random commit record
all_records = list(adapter.stream_records(CSV_PATH))
random_record = random.choice(all_records)

# 4. Extract rich payload parameters from repository metadata
git_payload = loader.fetch_commit_data(
    project=random_record["project"], 
    commit_id=random_record["commit_id"]
)

if git_payload:
    full_payload = {**random_record, **git_payload}
    
    # 5. Print out the raw dictionary dynamically via our class utility
    CommitPayloadPrinter.print_payload(full_payload)
    
    # 6. Parse and check entity configurations
    parsed_results = parser.parse(full_payload)

✅ Success! Raw payload retrieved with customized features.
KEY                       | VALUE
commit_id                 | 502bbf1201218684135d0b12aed12288b13a4ed8
project                   | apache/groovy
buggy                     | True
fix                       | True
year                      | 2013
author_date               | 1374259802
message                   | GROOVY-5873: generalize generics application and extend to be used on fields ...
diff                      | [Length: 9792 chars] -> @@ -1002,8 +1002,14 @@ public class StaticTypeCheckingVisitor extends ClassCodeVisitorSupport {     ...
parents                   | ['4064d24140443dca431143b2d3b45c49de5c5c75']
parents_length            | 1
linked_issues             | ['GROOVY-5873']
containing_branches       | ['master']
author_name               | Jochen Theodorou
author_email              | blackdrag@gmx.org
authored_timestamp        | 1374259802
authored_datetime         | 2013-07-19T20:50:02+02:00
committer_name         

In [12]:
# 1. Initialize our components
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# 2. Test reading records via the Adapter
print("--- Testing CSV Adapter Filtering ---")
records_stream = adapter.stream_records(CSV_PATH)

# Take the first two rows for validation
for i, record in enumerate(records_stream):
    if i >= 2: 
        break
    print(f"\nRecord #{i+1} parsed from CSV:")
    print(record)
    
    # 3. Use the filtered metadata to fetch Git info right away
    print(f"Fetching Git text data for commit: {record['commit_id']}...")
    git_payload = loader.fetch_commit_data(project=record['project'], commit_id=record['commit_id'])
    
    if git_payload:
        print(f"✅ Extracted Message length: {len(git_payload['message'])} chars")
        print(f"✅ Extracted Diff length: {len(git_payload['diff'])} chars")

--- Testing CSV Adapter Filtering ---

Record #1 parsed from CSV:
{'commit_id': '7b8480744ea6e6fb41efd4329bb470c8f3c763db', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1070355653'}
Fetching Git text data for commit: 7b8480744ea6e6fb41efd4329bb470c8f3c763db...
✅ Extracted Message length: 190 chars
✅ Extracted Diff length: 15148 chars

Record #2 parsed from CSV:
{'commit_id': '192b631e7be302ecde822546ba70a9853ddbda01', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1063298262'}
Fetching Git text data for commit: 192b631e7be302ecde822546ba70a9853ddbda01...
✅ Extracted Message length: 135 chars
✅ Extracted Diff length: 613 chars


# Gitpython Commit Object

In [13]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# Grab a random commit to inspect
record = random.choice(list(adapter.stream_records(CSV_PATH)))
repo = loader._get_repo(record["project"])
commit_obj = repo.commit(record["commit_id"])

print(f"🔬 Dissecting GitPython Commit Object for ID: {commit_obj.hexsha}\n")
print("=" * 60)

🔬 Dissecting GitPython Commit Object for ID: 5b7b4ab6fc896f6cfcefec978d38dabdd5047dae



In [14]:
properties_list = []
methods_list = []

# Analyze every attribute on the live object
for name, value in inspect.getmembers(commit_obj):
    if name.startswith('_'): 
        continue  # Skip private attributes
        
    try:
        if inspect.ismethod(value) or inspect.isroutine(value):
            methods_list.append(name)
        else:
            properties_list.append((name, type(value).__name__))
    except Exception:
        properties_list.append((name, "Unknown/Unloaded Property"))

print("📋 DATA FIELDS & PROPERTIES AVAILABLE:")
print("-" * 40)
for prop, data_type in sorted(properties_list):
    print(f"  {prop:<25} [Type: {data_type}]")

print("\n⚙️ EXECUTABLE METHODS AVAILABLE:")
print("-" * 40)
for method in sorted(methods_list):
    print(f"  {method}()")

📋 DATA FIELDS & PROPERTIES AVAILABLE:
----------------------------------------
  INDEX                     [Type: DiffConstants]
  Index                     [Type: DiffConstants]
  NULL_BIN_SHA              [Type: bytes]
  NULL_HEX_SHA              [Type: str]
  NULL_TREE                 [Type: DiffConstants]
  TIobj_tuple               [Type: _GenericAlias]
  TYPES                     [Type: tuple]
  author                    [Type: Actor]
  author_tz_offset          [Type: int]
  authored_date             [Type: int]
  authored_datetime         [Type: datetime]
  binsha                    [Type: bytes]
  co_authors                [Type: list]
  committed_date            [Type: int]
  committed_datetime        [Type: datetime]
  committer                 [Type: Actor]
  committer_tz_offset       [Type: int]
  conf_encoding             [Type: str]
  data_stream               [Type: OStream]
  default_encoding          [Type: str]
  encoding                  [Type: str]
  env_author_dat

# Parsers

In [15]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = RegexCommitParser()

print(f"Reading records from {CSV_PATH}...")
all_records = list(adapter.stream_records(CSV_PATH))

if not all_records:
    print("❌ No records found in the metadata file.")
else:
    random_record = random.choice(all_records)
    print(f"🎲 Randomly selected commit ID: {random_record['commit_id']} from {random_record['project']}")
    
    print("Extracting payload from local git repository...")
    git_payload = loader.fetch_commit_data(
        project=random_record["project"], 
        commit_id=random_record["commit_id"]
    )
    
    if git_payload:
        full_payload = {**random_record, **git_payload}
        
        print("\n=================== RAW COMMIT DIFF ===================")
        print(full_payload.get("diff", "No diff available for this commit."))
        print("========================================================\n")
        
        print("Executing simultaneous parse cycle...")
        results = parser.parse(full_payload)
        
        print("\n=== Extracted Knowledge Graph Entities (Random Sample) ===")
        print(json.dumps(results, indent=10))
    else:
        print("❌ Could not extract data from the Git repository. Verify your local paths match.")

Reading records from E:/Projects/kgcommit/data/apachejit/projects/apache_groovy.csv...
🎲 Randomly selected commit ID: 4ef6328facf58c73897cfdfdcab277b98ab67825 from apache/groovy
Extracting payload from local git repository...

=================== RAW COMMIT DIFF ===================
@@ -195,7 +195,8 @@ public class Node {
                 Object child = iter.next();
                 if (child instanceof Node) {
                     Node childNode = (Node) child;
-                    if (key.equals(childNode.name())) {
+                    Object childNodeName = childNode.name();
+                    if (childNodeName != null && childNodeName.equals(key)) {
                         answer.add(childNode);
                     }
                 }

@@ -180,7 +180,7 @@ public class QName implements Serializable {
     /**
      * Tests this QName for equality with another object.
      * <p>
-     * If the given object is not a QName or is null then this method
+     * If the given object i